# Tools 

- **Problems with LLMs** 
    - LLMs are powerful reasoning engines but they are fundamentally stateless text predictors. 
    - On their own they cannot fetch live data, run code, query a database, call an API, or interact with external systems. 
    - They are frozen in time — limited to what they learned during training.

- **Solution**
    - Tools solve this by giving the LLM a set of functions it can call at runtime. 
    - The LLM decides WHEN and with WHAT arguments to call a tool — you decide WHAT the tool does.

- **How it works?**
    - User asks:  'What is the weather in Dubai right now?'
 
    1. LLM receives the question + list of available tools
    2. LLM decides: I need the 'get_weather' tool with input 'Dubai'
    3. LangChain calls get_weather('Dubai') → returns '38°C, sunny'
    4. LLM receives the tool result
    5. LLM generates final answer: 'It is currently 38°C and sunny in Dubai.'

- **Key Insight:**  The LLM does not execute the tool — it only decides to call it and with what arguments. LangChain's runtime executes the actual function.


# Langchain tools

- LangChain ships with a rich library of pre-built tools for common tasks. 
- These are production-tested and ready to use out of the box.

## Wiki tool example

In [9]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.tools import Tool   # Updated import for LangChain 1.x

# Step 1: Initialize Wikipedia tool
wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

wikipedia_tool = Tool(
    name="Wikipedia",
    func=wiki.run,   # callable function for the tool
    description="Useful for fetching facts from Wikipedia"
)

# Step 2: Call tools directly (no agent)
query_1 = "Langchain"
result_1 = wikipedia_tool.run(query_1)

print("\nWikipedia Tool Result:\n")
print(result_1)


Wikipedia Tool Result:

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.



Page: Model Context Protocol
Summary: The Model Context Protocol (MCP) is an open standard and open-source framework introduced by Anthropic in November 2024 to standardize the way artificial intelligence (AI) systems like large language models (LLMs) integrate and share data with external tools, systems, and data sources. MCP provides a universal interface for reading files, executing functions, and handling contextual prompts. Following its announcement, the protocol was adopted by major AI providers, including OpenAI and Google DeepMind.

Page: Vector database
Summary: A vector database, vector store or vector sear

## Simple Calculator tool

In [7]:
calculator_tool = Tool(
    name="Calculator",
    func=lambda x: eval(x),  # simple math evaluator - tells what function should run when the tool is called
    description="Useful for running math expressions like '3+12'"
)

# Call tools directly (no agent)
query_2 = "12 * 7 + 5"
result_2 = calculator_tool.run(query_2)

# Step 4: Print results
print("\nCalculator Tool Result:\n")
print(result_2)



Calculator Tool Result:

89


## Duckduckgo search


In [12]:
from langchain_community.tools import DuckDuckGoSearchRun
search = DuckDuckGoSearchRun()
result = search.invoke('latest AI developments 2025')
print(result)

The latest McKinsey Global Survey on the state of AI reveals a landscape defined by both wider use—including growing proliferation of agentic AI ... The latest AI developments in 2024- 2025 are driving transformation across industries, creating massive opportunities for entrepreneurs willing to ... One of the most innovative AI Developments in 2025 is the fast-growing popularity of AI -powered robotics, which are not simply machines that can run ... From reasoning-first models and creative tools to new safety rules and silicon super-leaps, here are 25 developments shaping 2025 —curated for ... The Growing Significance of AI Development in 2025 Why Developers Are Adopting AI Tools: Leading AI Tools for Developers in 2025 1.


## REPL tool - Execute python code 

In [13]:
!pip install langchain_experimental

In [14]:
from langchain_experimental.tools import PythonREPLTool
 
python_repl = PythonREPLTool()
 
# LLM can write and execute Python code
result = python_repl.invoke('print([x**2 for x in range(1,6)])')
print(result)  # [1, 4, 9, 16, 25]

Python REPL can execute arbitrary code. Use with caution.


[1, 4, 9, 16, 25]



# @tool decorator - Simplest way

- The @tool decorator is the fastest way to turn any Python function into a LangChain tool. 
- The function name becomes the tool name, and the docstring becomes the tool description — this description is what the LLM reads to decide when to use the tool.
- Docstring = Tool Description:  Write clear, specific docstrings. The LLM reads this to decide WHEN to use your tool. Vague descriptions lead to wrong tool selection.

In [3]:
from langchain_core.tools import tool
 
# The docstring is CRITICAL — it tells the LLM what this tool does
@tool
def get_word_length(word: str) -> int:
    """Read the input and returns the number of words in a sentence."""
    return len(word.split())

# Inspect the tool
print(get_word_length.name)         # get_word_length
print(get_word_length.description)  # Read the input and returns the number of words in a sentence.
print(get_word_length.args)         # {'word': {'title': 'Word', 'type': 'string'}}
 
# Call it directly (for testing)
print(get_word_length.invoke('LangChain is great'))


get_word_length
Read the input and returns the number of words in a sentence.
{'word': {'title': 'Word', 'type': 'string'}}
3


# @tool with multiple inputs

In [15]:
from langchain_core.tools import tool
 
@tool
def calculate_bmi(weight_kg: float, height_m: float) -> str:
    """Calculates Body Mass Index (BMI) given weight in kg and height in metres.
    Returns the BMI value and category (underweight/normal/overweight/obese)."""
    bmi = weight_kg / (height_m ** 2)
    if bmi < 18.5:   category = 'Underweight'
    elif bmi < 25:   category = 'Normal'
    elif bmi < 30:   category = 'Overweight'
    else:            category = 'Obese'
    return f'BMI: {bmi:.1f} ({category})'
 
result = calculate_bmi.invoke({'weight_kg': 70, 'height_m': 1.75})
print(result)  


BMI: 22.9 (Normal)
